# Watermark U-Net — Colab run (warm restart from v2 weights)

Free Colab: GPU not guaranteed, sessions die (12h cap + ~90min idle).
Survival rules: outputs live on Drive, `--resume` continues after death,
run any cell every 1-2h against idle disconnect. Download best weights regularly.

In [ ]:
# ===== CELL 0 — parameters (edit ONLY here) + GPU + Drive + disk gate =====
EPOCHS, BATCH, SIZE, LR, PATIENCE = 80, 12, 384, 2e-4, 15
USE_LOGO = False  # True only with 20GB+ free (15GB download)
N_SYNTH = 5000
MIN_FREE_GB = 8
DRIVE_OUT = '/content/drive/MyDrive/hamrah-watermark/models'
import shutil
import torch
print('cuda:', torch.cuda.is_available())
from google.colab import drive
drive.mount('/content/drive')
free_gb = shutil.disk_usage('/content').free / 1e9
drive_free = shutil.disk_usage('/content/drive').free / 1e9
print(f'local free: {free_gb:.1f} GB | drive free: {drive_free:.1f} GB')
assert torch.cuda.is_available(), 'STOP: no GPU — Runtime → Change runtime type → GPU'
assert free_gb >= MIN_FREE_GB, f'STOP: need {MIN_FREE_GB}GB local free'
assert drive_free >= 5, 'STOP: need 5GB free on Drive'
print('gate OK — continue')

In [ ]:
# ===== CELL 1 — repo + deps (re-runnable) =====
import os
os.chdir('/content')
get_ipython().system('test -d hamrah-watermark/.git && (cd hamrah-watermark && git pull) || git clone https://github.com/AliTabibAzar/hamrah-watermark.git hamrah-watermark')
os.chdir('/content/hamrah-watermark')
get_ipython().system('pip install -q -r requirements-train.txt albumentations rapidocr-onnxruntime gdown')

In [ ]:
# ===== CELL 2 — latest weights live on Drive (verified). Priority: Drive > local > none. =====
# One-time browser setup: put the healthy watermark-unet.pt (~98MB) in
# Drive at MyDrive/hamrah-watermark/models/. After that, never upload again.
import os
os.makedirs('models', exist_ok=True)
def _healthy(path):
    import torch
    try:
        if os.path.getsize(path) < 50000000:
            return False, 'too small, truncated download?'
        sd = torch.load(path, map_location='cpu')
        return (True, f'OK, {len(sd)} tensors') if isinstance(sd, dict) else (False, 'not a state dict')
    except Exception as e:
        return False, str(e)[:120]
drive_w = f'{DRIVE_OUT}/watermark-unet.pt'
local_w = 'models/watermark-unet.pt'
if os.path.exists(drive_w):
    ok, msg = _healthy(drive_w)
    print(f'Drive weights: {msg}')
    assert ok, 'STOP: Drive weights corrupt — re-upload the healthy file and rerun'
    get_ipython().system(f'cp {drive_w} {local_w}')
    print('synced Drive -> local models/')
elif os.path.exists(local_w):
    ok, msg = _healthy(local_w)
    print(f'local weights: {msg}')
    assert ok, 'STOP: local weights corrupt — upload the healthy ~98MB file and rerun'
    get_ipython().system(f'mkdir -p {DRIVE_OUT} && cp {local_w} {drive_w}')
    print('seeded local -> Drive; future sessions need no upload')
else:
    print('NO weights anywhere — fresh start from random init.')

In [ ]:
# ===== CELL 3 — CLWD train data (skip if converted) =====
import os
need = not (os.path.isdir('data/clwd_train/images') and len(os.listdir('data/clwd_train/images')) > 50000)
print('CLWD-train needed:', need)
if need:
    get_ipython().system('test -f clwd.zip || gdown --fuzzy "https://drive.google.com/file/d/17y1gkUhIV6rZJg1gMG-gzVMnH27fm4Ij/view?usp=sharing" -O clwd.zip')
    get_ipython().system("unrar x clwd.zip 'CLWD/train/Mask/*' data/clwd_train_raw/ || sudo apt-get install -y -qq unrar")
    get_ipython().system("unrar x clwd.zip 'CLWD/train/Watermarked_image/*' data/clwd_train_raw/")
    get_ipython().system('python scripts/convert_clwd.py --src data/clwd_train_raw/CLWD/train --dst data/clwd_train')
    get_ipython().system('rm -rf data/clwd_train_raw')
get_ipython().system('ls data/clwd_train/images | wc -l')

In [ ]:
# ===== CELL 4 — CLWD test data for the eval gate (skip if converted) =====
import os
need = not (os.path.isdir('data/clwd_test/images') and len(os.listdir('data/clwd_test/images')) > 5000)
print('CLWD-test needed:', need)
if need:
    get_ipython().system('test -f clwd.zip || gdown --fuzzy "https://drive.google.com/file/d/17y1gkUhIV6rZJg1gMG-gzVMnH27fm4Ij/view?usp=sharing" -O clwd.zip')
    get_ipython().system("unrar x clwd.zip 'CLWD/test/*' data/clwd_raw/")
    get_ipython().system('python scripts/convert_clwd.py --src data/clwd_raw/CLWD/test --dst data/clwd_test')
    get_ipython().system('rm -rf data/clwd_raw')
get_ipython().system('ls data/clwd_test/images | wc -l')

In [ ]:
# ===== CELL 5 — synthetic top-up (skip if present) =====
import os
need = not (os.path.isdir('data/synth/images') and len(os.listdir('data/synth/images')) >= N_SYNTH)
print('synth needed:', need)
if need:
    get_ipython().system(f'python scripts/gen_synthetic.py --bg data/clwd_train/images --out data/synth --n {N_SYNTH}')

In [ ]:
# ===== CELL 6 — TRAIN on Drive (warm restart from latest). STOP rule below. =====
# STOP rule: val IoU < 0.3 past epoch 15 -> stop cell, send the log.
# Keep-alive: run any cell every 1-2h. Download Drive best regularly.
import os
w = f'{DRIVE_OUT}/watermark-unet.pt'
get_ipython().system(f'python -c "import torch; sd=torch.load(\'{w}\', map_location=\'cpu\'); print(\'resume source OK,\', len(sd), \'tensors\')"' if os.path.exists(w) else 'echo "no Drive weights — fresh start"')
get_ipython().system(f'python train_unet.py --data data/clwd_train data/synth --out {DRIVE_OUT} --epochs {EPOCHS} --batch {BATCH} --size {SIZE} --lr {LR} --patience {PATIENCE} --scheduler cosine --resume')

In [ ]:
# ===== CELL 7 — eval gate (CLWD must PASS) =====
# Bars: CLWD IoU>=0.45 + recall>=0.70.
get_ipython().system(f'python scripts/eval_bulk.py --data data/clwd_test --mode ensemble --limit 500 --thresholds 0.3 0.5 0.7 --weights {DRIVE_OUT}/watermark-unet.pt')

In [ ]:
# ===== CELL 8 — export checklist (DO THIS before the session dies) =====
# [ ] best weights downloaded locally AND added to the Kaggle dataset as watermark-unet-v2.pt
# [ ] epoch log copied (best val IoU + epoch number)
# [ ] eval GATE line + table sent for review
get_ipython().system(f'ls -la {DRIVE_OUT}/*.pt')